In [ ]:
import pandas as pd
import numpy as np

# Da Citi Bike riesige CSV-Dateien anbietet, simulieren wir hier den Ladevorgang 
# mit dem typischen Dateinamen, den du dir von der Citi Bike Website lädst.
# Beispiel: "202305-citibike-tripdata.csv"
file_path = "202305-citibike-tripdata.csv"

# Wir laden die Daten (Extract):

In [ ]:
# Wir laden die CSV-Datei in einen Pandas DataFrame
# (Tipp: low_memory=False verhindert Warnungen bei gemischten Datentypen in großen Dateien)
try:
    df_citi = pd.read_csv(file_path, low_memory=False)
    print("Erfolg! Citi Bike Daten geladen.")
    print("Die ersten 5 Zeilen der Rohdaten:")
    display(df_citi.head())
except FileNotFoundError:
    print(f"Fehler: Datei {file_path} nicht gefunden. Bitte lade sie in denselben Ordner wie das Notebook.")

# Wir schauen uns die Datentypen an:

In [ ]:
print("Datentypen der einzelnen Spalten:")
print(df_citi.dtypes)

print("\nZusätzliche Infos (Anzahl der Zeilen und Non-Null Werte):")
df_citi.info()

# Wir überprüfen die Daten auf mögliche Verunreinigungen:

In [ ]:
print("Anzahl der fehlenden Werte (NaN) pro Spalte:")
# Meistens fehlen bei Citi Bike ein paar Start/End-Stationen, weil GPS-Aussetzer auftraten
print(df_citi.isnull().sum())

print(f"\nAnzahl der komplett identischen Duplikate (Sollte 0 sein): {df_citi.duplicated().sum()}")

# Wir bereinigen die Daten (Transform - Teil 1: Cleaning):

In [ ]:
# 1. Zeilen löschen, bei denen die Start-Station fehlt (Ohne Start-Station kein lokales Marketing!)
df_citi.dropna(subset=['start_station_id', 'start_station_name'], inplace=True)

# 2. Zeit-Strings in echte datetime-Objekte umwandeln
df_citi['started_at'] = pd.to_datetime(df_citi['started_at'])
df_citi['ended_at'] = pd.to_datetime(df_citi['ended_at'])

# 3. Wir berechnen die Fahrtdauer in Minuten (als neues Measure)
df_citi['Duration_Min'] = (df_citi['ended_at'] - df_citi['started_at']).dt.total_seconds() / 60

# 4. Filterung von fehlerhaften Daten: 
# Fahrten unter 1 Minute (meistens kaputte Räder, die sofort zurückgestellt werden) 
# oder über 24 Stunden (1440 Min, gestohlene/vergessene Räder) werfen wir raus.
df_citi = df_citi[(df_citi['Duration_Min'] > 1) & (df_citi['Duration_Min'] < 1440)]

print("Bereinigung abgeschlossen. Übrige Zeilen:", len(df_citi))

In [ ]:
# 1. UMSATZ-BERECHNUNG (Business Case):
# Annahme laut offiziellem Citi Bike Pricing: Casuals zahlen $4.49 Freischaltung + $0.26 pro Minute.
# Members betrachten wir in der Einzel-Fahrt als $0 Umsatz (da sie pauschal $219/Jahr zahlen).
df_citi['Revenue_USD'] = np.where(
    df_citi['member_casual'] == 'casual', 
    4.49 + (df_citi['Duration_Min'] * 0.26), 
    0.0
)

# 2. FOREIGN KEYS FÜR POWER BI ERSTELLEN:
# Date_ID (Format: YYYYMMDD)
df_citi['Date_ID'] = df_citi['started_at'].dt.strftime('%Y%m%d').astype(int)

# Hour_ID (Format: 0 bis 23)
df_citi['Hour_ID'] = df_citi['started_at'].dt.hour

print("Umsatz und Foreign Keys wurden berechnet!")
display(df_citi[['started_at', 'member_casual', 'Duration_Min', 'Revenue_USD', 'Date_ID', 'Hour_ID']].head())

# Wir laden sie (Staging):

In [ ]:
# Wir speichern die bereinigten, angereicherten "flachen" Rohdaten als Zwischenspeicher ab.
staging_pfad = "citibike_cleaned_staging.csv"
df_citi.to_csv(staging_pfad, index=False)
print(f"Bereinigte Zwischendaten gespeichert unter: {staging_pfad}")

#Load - Das Star Schema aufbauen:

In [ ]:
# JETZT BAUEN WIR DIE EXAKTEN TABELLEN FÜR POWER BI!

# 1. Dimension: Dim_Station erstellen und speichern
# Wir ziehen uns nur die eindeutigen (unique) Stationen aus den Rohdaten
Dim_Station = df_citi[['start_station_id', 'start_station_name', 'start_lat', 'start_lng']].drop_duplicates()
Dim_Station.rename(columns={
    'start_station_id': 'StartStation_ID',
    'start_station_name': 'Station_Name',
    'start_lat': 'Latitude',
    'start_lng': 'Longitude'
}, inplace=True)
Dim_Station.to_csv("Dim_Station.csv", index=False)

# 2. Dimension: Dim_UserType erstellen und speichern
Dim_UserType = pd.DataFrame({
    'UserType_ID': [1, 2],
    'User_Type_Name': ['casual', 'member']
})
Dim_UserType.to_csv("Dim_UserType.csv", index=False)

# 3. Dimension: Dim_RideableType erstellen und speichern
unique_bikes = df_citi['rideable_type'].unique()
Dim_RideableType = pd.DataFrame({
    'RideableType_ID': range(1, len(unique_bikes) + 1),
    'Rideable_Type_Name': unique_bikes
})
Dim_RideableType.to_csv("Dim_RideableType.csv", index=False)

# 4. Dimension: Dim_Hour erstellen und speichern
Dim_Hour = pd.DataFrame({'Hour_ID': range(0, 24)})
# Kategorisierung (Vormittag, Nachmittag etc.) für das Dashboard
Dim_Hour['Time_of_Day'] = pd.cut(
    Dim_Hour['Hour_ID'], 
    bins=[-1, 5, 11, 17, 21, 24], 
    labels=['Nacht', 'Morgen', 'Nachmittag', 'Abend', 'Nacht'], ordered=False
)
Dim_Hour.to_csv("Dim_Hour.csv", index=False)

print("Alle 4 Dimensionstabellen wurden erfolgreich erstellt und als CSV gespeichert!")

Load - Die finale Fakten-Tabelle

In [ ]:
# ZUR ERINNERUNG: Die Faktentabelle darf nur IDs und Zahlen (Measures) enthalten!

# Wir mergen unsere Rohdaten mit unseren neu erstellten kleinen Dimensionen, 
# um die UserType_ID und RideableType_ID zu bekommen.
df_facts = df_citi.merge(Dim_UserType, left_on='member_casual', right_on='User_Type_Name', how='left')
df_facts = df_facts.merge(Dim_RideableType, left_on='rideable_type', right_on='Rideable_Type_Name', how='left')

# Jetzt filtern wir nur die Spalten heraus, die für die Faktentabelle Fact_Rides vorgesehen sind:
Fact_Rides = df_facts[[
    'Date_ID', 
    'Hour_ID', 
    'start_station_id', 
    'UserType_ID', 
    'RideableType_ID', 
    'Duration_Min', 
    'Revenue_USD'
]].copy()

# Saubere Benennung für Power BI
Fact_Rides.rename(columns={'start_station_id': 'StartStation_ID'}, inplace=True)

# Wir speichern die finale Faktentabelle ab
Fact_Rides.to_csv("Fact_Rides.csv", index=False)

print("VOILÀ! Die finale Fact_Rides Tabelle ist fertig und sieht so aus:")
display(Fact_Rides.head())
print(f"Anzahl der Zeilen in den Fakten: {len(Fact_Rides)}")